In [ ]:
# Kaggle push sends only the notebook body, so recreate local support files first.
import base64 as _kaggle_base64
import hashlib as _kaggle_hashlib
import io as _kaggle_io
import zipfile as _kaggle_zipfile
from pathlib import Path as _KagglePath

_KAGGLE_SUPPORT_ZIP_B64 = (
    '...'
    'BgAAAAAGAAYAcgEAAPk1AAAAAA=='
)
_KAGGLE_SUPPORT_MANIFEST = {
  "config.yaml": {
    "bytes": 6001,
    "sha256": "8c95d82c37601a03be8955d21b1ac92662053b80dd12953034c90929bb337b53"
  },
  "linear_md_z_prior_global_search.py": {
    "bytes": 35951,
    "sha256": "23725ed9c47caf4b6816d28656bc5da7d836d8ff752c988df6d32b59bf819f5e"
  },
  "project.yml": {
    "bytes": 1067,
    "sha256": "7c933e0240d1b4a8f66a4085bcb9b1eb97d9100f1ac494c8bb789219d39fe157"
  },
  "settings.py": {
    "bytes": 10477,
    "sha256": "1f793b41746418c42e732a5dfc64eace8caea247f00151c1ba004e01acdd9861"
  },
  "src/__init__.py": {
    "bytes": 47,
    "sha256": "d531b37e210d810fc44638dd3bc6e6ffbf24223f9b1894c893f45a6ee4622e64"
  },
  "src/paths.py": {
    "bytes": 233,
    "sha256": "5aff21dc5fd8725576e38be7da9e222d25f2aa02237b8e445d2c240c53305436"
  }
}
_zip_bytes = _kaggle_base64.b64decode(_KAGGLE_SUPPORT_ZIP_B64)
with _kaggle_zipfile.ZipFile(_kaggle_io.BytesIO(_zip_bytes)) as _zip:
    for _member in _zip.infolist():
        _path = _KagglePath(_member.filename)
        if _path.is_absolute() or '..' in _path.parts:
            raise RuntimeError(f'unsafe bootstrap path: {_member.filename}')
    _zip.extractall('.')
for _relative_path, _info in _KAGGLE_SUPPORT_MANIFEST.items():
    _contents = _KagglePath(_relative_path).read_bytes()
    _digest = _kaggle_hashlib.sha256(_contents).hexdigest()
    if _digest != _info['sha256']:
        raise RuntimeError(f'bootstrap hash mismatch: {_relative_path}')
print(
    f"Prepared {len(_KAGGLE_SUPPORT_MANIFEST)} Kaggle support files "
    "from zip bootstrap."
)


# exp113_linear_md_z_prior_global_search train

Fold-safe global search for a weak linear MD/Z TVT prior on exp099 pseudo-tail rows.

## Contents

1. Setup and configuration
2. Input artifact checks
3. Linear MD/Z prior grid and fold selection
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from linear_md_z_prior_global_search import (
    EXP065_CLUSTER_ASSIGNMENTS,
    EXP099_FEATURE_CACHE,
    find_artifact,
    parse_candidates,
    parse_group_methods,
    run_audit,
)

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
candidates = parse_candidates(config)
group_methods = parse_group_methods(config)

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Candidate count:", len(candidates))
print("First candidates:", [candidate.name for candidate in candidates[:8]])
print("Group methods:", [method.name for method in group_methods])

## 2. Input artifact checks

In [ ]:
feature_cache = find_artifact(
    EXP099_FEATURE_CACHE,
    get_nested(config, "data.exp099_train_feature_cache_local"),
)
cluster_assignments = find_artifact(
    EXP065_CLUSTER_ASSIGNMENTS,
    get_nested(config, "data.exp065_cluster_assignments_local"),
)

feature_header = pd.read_csv(feature_cache, nrows=0).columns.tolist()
cluster_preview = pd.read_csv(cluster_assignments, nrows=5)
sample_train_files = sorted(paths.train_data_dir.glob("*__horizontal_well.csv"))[:3]

print("Feature cache:", feature_cache)
print("Feature columns:", feature_header)
print("Cluster assignments:", cluster_assignments)
print("Sample train files:", sample_train_files)
display(cluster_preview)

## 3. Linear MD/Z prior grid and fold selection

In [ ]:
summary = run_audit(config=config, paths=paths)
print(json.dumps(summary["fold_selected_linear"], indent=2, sort_keys=True))
print("Delta fold-selected linear - likPF RMSE:", summary["delta_fold_selected_minus_likpf_rmse"])
print("GR sample rows:", summary["gr_loss"]["sample_rows"])

## 4. Metrics and artifacts

In [ ]:
metrics_path = Path(summary["artifacts"]["candidate_metrics"])
fold_path = Path(summary["artifacts"]["fold_selection"])
bucket_path = Path(summary["artifacts"]["bucket_metrics"])
by_well_path = Path(summary["artifacts"]["by_well"])
group_path = Path(summary["artifacts"]["group_consistency"])

metrics = pd.read_csv(metrics_path)
fold_selection = pd.read_csv(fold_path)
bucket_metrics = pd.read_csv(bucket_path)
by_well = pd.read_csv(by_well_path)
group_consistency = pd.read_csv(group_path)

display(metrics.head(30))
display(fold_selection)
display(bucket_metrics[bucket_metrics["candidate"].isin(metrics.head(5)["candidate"])].head(60))
display(by_well.sort_values("rmse", ascending=False).head(20))
display(group_consistency.sort_values(["mode_share", "wells_with_scores"], ascending=[False, False]).head(20))

print("Summary:", summary["artifacts"]["summary"])
print("OOF predictions:", summary["artifacts"]["oof_predictions"])
print("Feature schema:", summary["artifacts"]["feature_schema"])